# Update Prices

This notebook pulls the day-ahead electricity prices from ENTSO-E for all 5 zones and writes them to the clean price CSVs. It extends the previous group's data (which only went up to November 2025) and also pulls the full history for the two new zones DE-LU and FR from scratch.

The output format matches the previous group's preprocessed files (wide format, one row per day,columns h00 to h23).

Setting up the ENTSO-E client. Own (free) API token needs to be pasted below.

In [ ]:
from entsoe import EntsoePandasClient
from pathlib import Path
import pandas as pd

TOKEN = "add_your_entsoe_api_key_here"  # replace with your own ENTSO-E API key
client = EntsoePandasClient(api_key=TOKEN)

Zones and their ENTSO-E codes. We pull from 2022 to today to match our training window.

In [ ]:
ZONES = {
    'DK1': 'DK_1',
    'NO2': 'NO_2',
    'ES':  'ES',
    'DE_LU': 'DE_LU',
    'FR': 'FR'
}

start = pd.Timestamp('20220101', tz='Europe/Brussels')
end   = pd.Timestamp.now(tz='Europe/Brussels')

Fetching the prices for each zone. ENTSO-E returns quarter-hourly data for some zones so we resample to hourly by just taking the first entry of each hour (as we agreed in the team). We also handle the DST duplicate hours. If a zone already has a file we just append the new dates, if not (the new zones) we create it from scratch.

In [ ]:
for zone_name, zone_code in ZONES.items():
    print(f"Fetching {zone_name}...")
    prices = client.query_day_ahead_prices(zone_code, start=start, end=end)
    
    # convert to dataframe and reshape to match previous group's format
    df = prices.to_frame(name='price')
    df.index = df.index.tz_convert('Europe/Vienna')
    df = df.resample('h').mean()
    df = df[~df.index.duplicated(keep='first')]
    df['date'] = df.index.date
    df['hour'] = df.index.hour
    df = df[~df.duplicated(subset=['date', 'hour'], keep='first')]
    df_wide = df.pivot(index='date', columns='hour', values='price')    
    df_wide.columns = [f'h{str(h).zfill(2)}' for h in df_wide.columns]
    
    # load existing preprocessed CSV and append
    existing_path = f'data/clean/{zone_name}_preprocessed.csv'
    
    if Path(existing_path).exists():
        df_existing = pd.read_csv(existing_path, index_col=0)
        df_combined = pd.concat([df_existing, df_wide])
        df_combined = df_combined[~df_combined.index.duplicated(keep='last')]
    else:
        # new zone, no existing file
        df_combined = df_wide
    
    df_combined.to_csv(existing_path)
    print(f"  Saved {existing_path}")

print("Done.")